# 🎮 Steam Reviews — ML Model Eğitimi ve Değerlendirme

**Veri Kaynağı:** Delta Lake Gold Katmanı (`/delta/gold/features_table`)  
**Target:** `label` (0 veya 1)  
**Features:** `features` (VectorAssembler çıktısı — TF-IDF + sayısal özellikler)

---

## İçindekiler
1. Hazırlık (Spark, MLflow, Train/Test Split, Class Weight)  
2. Model Eğitimi (5 Model)  
3. Değerlendirme Metrikleri  
4. Karşılaştırma Tablosu  
5. Feature Importance  
6. En İyi Model Seçimi

---
## BÖLÜM 1 — Hazırlık

In [ ]:
# ── 1a. Kütüphaneler ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    NaiveBayes
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT

import mlflow
import mlflow.spark

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.style.use("seaborn-v0_8-whitegrid")

print("Kütüphaneler yüklendi ✓")

In [ ]:
# ── 1b. Spark oturumu ve Delta Lake'ten veri okuma ────────────────────────────────
spark = SparkSession.builder \
    .appName("SteamReviews_ML_Models") \
    .master("local[*]") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Gold katmandan features_table oku
DELTA_PATH = "/delta/gold/features_table"
df = spark.read.format("delta").load(DELTA_PATH)

print(f"Schema:")
df.printSchema()
print(f"\nToplam satır sayısı: {df.count():,}")
df.groupBy("label").count().show()

In [ ]:
# ── 1c. MLflow tracking URI ayarla ─────────────────────────────────────────────
mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("steam_reviews_classification")
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow Experiment: steam_reviews_classification")

In [ ]:
# ── 1d. Train/Test Split (%80/%20, seed=42) ───────────────────────────────────
train, test = df.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

print(f"Train set: {train.count():,} satır")
print(f"Test set:  {test.count():,} satır")
print(f"\nTrain sınıf dağılımı:")
train.groupBy("label").count().show()

In [ ]:
# ── 1e. Sınıf dengesizliği için weightCol hesapla ───────────────────────────────
# Her sınıf için ağırlık hesapla: toplam / (2 * sınıf_sayısı)
label_counts = train.groupBy("label").count().collect()
total_count = train.count()
num_classes = len(label_counts)

weight_dict = {}
for row in label_counts:
    weight_dict[row["label"]] = total_count / (num_classes * row["count"])

print("Sınıf ağırlıkları:")
for label, weight in sorted(weight_dict.items()):
    print(f"  Label {label}: weight = {weight:.4f}")

# weightCol kolonu ekle
weight_mapping = F.create_map([F.lit(x) for item in weight_dict.items() for x in item])
train = train.withColumn("classWeight", weight_mapping[F.col("label")])
test = test.withColumn("classWeight", weight_mapping[F.col("label")])

print("\nWeight kolonu eklendi ✓")
train.select("label", "classWeight").show(5)

---
## BÖLÜM 2 — Model Eğitimi (5 Model)

In [ ]:
# ── Yardımcı fonksiyonlar ──────────────────────────────────────────────────────
def evaluate_model(predictions):
    """Tüm metrikleri hesapla ve döndür."""
    # AUC-ROC
    binary_eval = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )
    auc = binary_eval.evaluate(predictions)
    
    # Accuracy
    multi_eval_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    accuracy = multi_eval_acc.evaluate(predictions)
    
    # F1
    multi_eval_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    f1 = multi_eval_f1.evaluate(predictions)
    
    # Precision
    multi_eval_prec = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
    )
    precision = multi_eval_prec.evaluate(predictions)
    
    # Recall
    multi_eval_rec = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="weightedRecall"
    )
    recall = multi_eval_rec.evaluate(predictions)
    
    return {"auc": auc, "accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}


def compute_confusion_matrix(predictions):
    """Manuel confusion matrix hesapla: TP, TN, FP, FN."""
    tp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
    tn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 0)).count()
    fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
    fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}


# Sonuçları saklamak için
results = []
all_predictions = {}
trained_models = {}

print("Yardımcı fonksiyonlar tanımlandı ✓")

In [ ]:
# ── MODEL 1: Logistic Regression ─────────────────────────────────────────────
print("="*60)
print("MODEL 1: Logistic Regression")
print("="*60)

with mlflow.start_run(run_name="LogisticRegression"):
    params = {"maxIter": 100, "regParam": 0.1, "elasticNetParam": 0.0}
    mlflow.log_params(params)
    
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        weightCol="classWeight",
        maxIter=100,
        regParam=0.1,
        elasticNetParam=0.0
    )
    
    start_time = time.time()
    model_lr = lr.fit(train)
    train_time = time.time() - start_time
    
    predictions_lr = model_lr.transform(test)
    metrics = evaluate_model(predictions_lr)
    cm = compute_confusion_matrix(predictions_lr)
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.spark.log_model(model_lr, "model")
    
    results.append({"Model": "Logistic Regression", **metrics, "Train Time": f"{train_time:.2f}s"})
    all_predictions["Logistic Regression"] = predictions_lr
    trained_models["Logistic Regression"] = model_lr
    
    print(f"  AUC:       {metrics['auc']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Train Time: {train_time:.2f}s")
    print(f"  Confusion Matrix: {cm}")
    print()

In [ ]:
# ── MODEL 2: Decision Tree Classifier ────────────────────────────────────────
print("="*60)
print("MODEL 2: Decision Tree Classifier")
print("="*60)

with mlflow.start_run(run_name="DecisionTree"):
    params = {"maxDepth": 10, "minInstancesPerNode": 5}
    mlflow.log_params(params)
    
    dt = DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        weightCol="classWeight",
        maxDepth=10,
        minInstancesPerNode=5
    )
    
    start_time = time.time()
    model_dt = dt.fit(train)
    train_time = time.time() - start_time
    
    predictions_dt = model_dt.transform(test)
    metrics = evaluate_model(predictions_dt)
    cm = compute_confusion_matrix(predictions_dt)
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.spark.log_model(model_dt, "model")
    
    results.append({"Model": "Decision Tree", **metrics, "Train Time": f"{train_time:.2f}s"})
    all_predictions["Decision Tree"] = predictions_dt
    trained_models["Decision Tree"] = model_dt
    
    print(f"  AUC:       {metrics['auc']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Train Time: {train_time:.2f}s")
    print(f"  Confusion Matrix: {cm}")
    print()

In [ ]:
# ── MODEL 3: Random Forest Classifier ────────────────────────────────────────
print("="*60)
print("MODEL 3: Random Forest Classifier")
print("="*60)

with mlflow.start_run(run_name="RandomForest"):
    params = {"numTrees": 100, "maxDepth": 8, "seed": 42}
    mlflow.log_params(params)
    
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        weightCol="classWeight",
        numTrees=100,
        maxDepth=8,
        seed=42
    )
    
    start_time = time.time()
    model_rf = rf.fit(train)
    train_time = time.time() - start_time
    
    predictions_rf = model_rf.transform(test)
    metrics = evaluate_model(predictions_rf)
    cm = compute_confusion_matrix(predictions_rf)
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.spark.log_model(model_rf, "model")
    
    results.append({"Model": "Random Forest", **metrics, "Train Time": f"{train_time:.2f}s"})
    all_predictions["Random Forest"] = predictions_rf
    trained_models["Random Forest"] = model_rf
    
    print(f"  AUC:       {metrics['auc']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Train Time: {train_time:.2f}s")
    print(f"  Confusion Matrix: {cm}")
    print()

In [ ]:
# ── MODEL 4: Gradient Boosted Trees (GBT) ────────────────────────────────────
print("="*60)
print("MODEL 4: Gradient Boosted Trees (GBT)")
print("="*60)

with mlflow.start_run(run_name="GBT"):
    params = {"maxIter": 50, "maxDepth": 6, "stepSize": 0.1}
    mlflow.log_params(params)
    
    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="label",
        weightCol="classWeight",
        maxIter=50,
        maxDepth=6,
        stepSize=0.1
    )
    
    start_time = time.time()
    model_gbt = gbt.fit(train)
    train_time = time.time() - start_time
    
    predictions_gbt = model_gbt.transform(test)
    metrics = evaluate_model(predictions_gbt)
    cm = compute_confusion_matrix(predictions_gbt)
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.spark.log_model(model_gbt, "model")
    
    results.append({"Model": "GBT", **metrics, "Train Time": f"{train_time:.2f}s"})
    all_predictions["GBT"] = predictions_gbt
    trained_models["GBT"] = model_gbt
    
    print(f"  AUC:       {metrics['auc']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Train Time: {train_time:.2f}s")
    print(f"  Confusion Matrix: {cm}")
    print()

In [ ]:
# ── MODEL 5: Naive Bayes ───────────────────────────────────────────────────────
# NOT: Naive Bayes negatif feature kabul etmez.
# Sadece TF-IDF + non-negative features kullanılır.
# Eğer features vektöründe negatif değer varsa, onları 0'a çekiyoruz.
print("="*60)
print("MODEL 5: Naive Bayes")
print("="*60)

from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, VectorUDT, DenseVector, SparseVector

def clip_negative_values(vec):
    """Vektördeki negatif değerleri 0'a çek."""
    if vec is None:
        return vec
    if isinstance(vec, SparseVector):
        new_values = np.maximum(vec.values, 0.0)
        return Vectors.sparse(vec.size, vec.indices, new_values.tolist())
    else:
        new_values = np.maximum(vec.toArray(), 0.0)
        return Vectors.dense(new_values.tolist())

clip_udf = udf(clip_negative_values, VectorUDT())

train_nb = train.withColumn("features_nn", clip_udf(F.col("features")))
test_nb = test.withColumn("features_nn", clip_udf(F.col("features")))

with mlflow.start_run(run_name="NaiveBayes"):
    params = {"smoothing": 1.0, "modelType": "multinomial"}
    mlflow.log_params(params)
    
    nb = NaiveBayes(
        featuresCol="features_nn",
        labelCol="label",
        weightCol="classWeight",
        smoothing=1.0,
        modelType="multinomial"
    )
    
    start_time = time.time()
    model_nb = nb.fit(train_nb)
    train_time = time.time() - start_time
    
    predictions_nb = model_nb.transform(test_nb)
    metrics = evaluate_model(predictions_nb)
    cm = compute_confusion_matrix(predictions_nb)
    
    mlflow.log_metrics(metrics)
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.spark.log_model(model_nb, "model")
    
    results.append({"Model": "Naive Bayes", **metrics, "Train Time": f"{train_time:.2f}s"})
    all_predictions["Naive Bayes"] = predictions_nb
    trained_models["Naive Bayes"] = model_nb
    
    print(f"  AUC:       {metrics['auc']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Train Time: {train_time:.2f}s")
    print(f"  Confusion Matrix: {cm}")
    print()

---
## BÖLÜM 3 — Değerlendirme Metrikleri (Detaylı Confusion Matrix)

In [ ]:
# ── Her model için detaylı confusion matrix ───────────────────────────────────
print("="*70)
print("CONFUSION MATRIX — TÜM MODELLER")
print("="*70)

cm_results = []
for model_name, preds in all_predictions.items():
    cm = compute_confusion_matrix(preds)
    cm_results.append({"Model": model_name, **cm})
    print(f"\n{model_name}:")
    print(f"  TP={cm['TP']:,}  FP={cm['FP']:,}")
    print(f"  FN={cm['FN']:,}  TN={cm['TN']:,}")

cm_df = pd.DataFrame(cm_results)
print("\n")
cm_df

---
## BÖLÜM 4 — Karşılaştırma Tablosu

In [ ]:
# ── Pandas DataFrame olarak karşılaştırma tablosu ───────────────────────────────
results_df = pd.DataFrame(results)
results_df = results_df[["Model", "auc", "accuracy", "f1", "precision", "recall", "Train Time"]]
results_df.columns = ["Model", "AUC", "Accuracy", "F1", "Precision", "Recall", "Train Time"]

# AUC'ye göre sırala
results_df = results_df.sort_values("AUC", ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("MODEL KARSILASTIRMA TABLOSU (AUC'ye göre sıralı)")
print("="*80)
results_df.style \
    .format({"AUC": "{:.4f}", "Accuracy": "{:.4f}", "F1": "{:.4f}", 
             "Precision": "{:.4f}", "Recall": "{:.4f}"}) \
    .background_gradient(cmap="Greens", subset=["AUC", "Accuracy", "F1"]) \
    .set_caption("Steam Reviews — 5 Model Karşılaştırması")

In [ ]:
# ── Metrik karşılaştırma bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = ["AUC", "Accuracy", "F1", "Precision", "Recall"]
x = np.arange(len(results_df))
width = 0.15

colors = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#e74c3c"]

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, results_df[metric], width, label=metric, color=colors[i])

ax.set_xlabel("Model")
ax.set_ylabel("Skor")
ax.set_title("Model Performans Karşılaştırması", fontsize=14)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df["Model"], rotation=15, ha="right")
ax.legend(loc="lower right", fontsize=10)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Baseline (0.5)")

plt.tight_layout()
plt.show()

---
## BÖLÜM 5 — Feature Importance

In [ ]:
# ── Random Forest Feature Importance ──────────────────────────────────────────
def plot_feature_importance(model, model_name, top_n=15):
    """Feature importance bar chart çiz."""
    importances = model.featureImportances.toArray()
    
    # Feature indekslerini al
    indices = np.argsort(importances)[::-1][:top_n]
    top_importances = importances[indices]
    top_labels = [f"Feature_{idx}" for idx in indices]
    
    # Metadata varsa feature isimlerini al
    try:
        attrs = train.schema["features"].metadata["ml_attr"]["attrs"]
        all_attrs = []
        for attr_type in ["numeric", "binary"]:
            if attr_type in attrs:
                all_attrs.extend(attrs[attr_type])
        if all_attrs:
            attr_dict = {a["idx"]: a["name"] for a in all_attrs}
            top_labels = [attr_dict.get(idx, f"Feature_{idx}") for idx in indices]
    except (KeyError, TypeError):
        pass
    
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(range(top_n), top_importances[::-1], color="#3498db", edgecolor="white")
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_labels[::-1], fontsize=10)
    ax.set_xlabel("Importance")
    ax.set_title(f"{model_name} — Top {top_n} Feature Importance", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return pd.DataFrame({"Feature": top_labels, "Importance": top_importances})


print("Random Forest — Top 15 Feature Importance:")
print("-" * 50)
rf_importance_df = plot_feature_importance(trained_models["Random Forest"], "Random Forest", top_n=15)
rf_importance_df

In [ ]:
# ── GBT Feature Importance ────────────────────────────────────────────────────
print("GBT — Top 15 Feature Importance:")
print("-" * 50)
gbt_importance_df = plot_feature_importance(trained_models["GBT"], "GBT", top_n=15)
gbt_importance_df

---
## BÖLÜM 6 — En İyi Model Seçimi

In [ ]:
# ── AUC'ye göre en iyi modeli seç ──────────────────────────────────────────────
best_model_name = results_df.iloc[0]["Model"]
best_auc = results_df.iloc[0]["AUC"]

print("="*60)
print(f"EN IYI MODEL: {best_model_name}")
print(f"AUC: {best_auc:.4f}")
print("="*60)

# MLflow'da best_model tag'i ekle
experiment = mlflow.get_experiment_by_name("steam_reviews_classification")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
best_run = runs.loc[runs["metrics.auc"].idxmax()]
best_run_id = best_run["run_id"]

with mlflow.start_run(run_id=best_run_id):
    mlflow.set_tag("best_model", "true")
    mlflow.set_tag("model_name", best_model_name)

print(f"\nMLflow'da '{best_model_name}' run'\u0131na 'best_model' tag'i eklendi ✓")
print(f"Run ID: {best_run_id}")

In [ ]:
# ── En iyi model — Confusion Matrix görselleştirme ─────────────────────────────
best_predictions = all_predictions[best_model_name]
cm = compute_confusion_matrix(best_predictions)

fig, ax = plt.subplots(figsize=(7, 6))

cm_matrix = np.array([[cm["TN"], cm["FP"]], [cm["FN"], cm["TP"]]])
im = ax.imshow(cm_matrix, cmap="Blues", interpolation="nearest")

# Değerleri hücrelere yaz
for i in range(2):
    for j in range(2):
        val = cm_matrix[i, j]
        color = "white" if val > cm_matrix.max() / 2 else "black"
        ax.text(j, i, f"{val:,}", ha="center", va="center",
                fontsize=16, fontweight="bold", color=color)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted 0", "Predicted 1"], fontsize=12)
ax.set_yticklabels(["Actual 0", "Actual 1"], fontsize=12)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")
ax.set_title(f"Confusion Matrix — {best_model_name}\n(AUC: {best_auc:.4f})", fontsize=14)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print(f"\nTP={cm['TP']:,}  FP={cm['FP']:,}")
print(f"FN={cm['FN']:,}  TN={cm['TN']:,}")

In [ ]:
# ── En iyi model — ROC Curve ───────────────────────────────────────────────────
from pyspark.ml.functions import vector_to_array

# Probability kolonundan pozitif sınıf olasılıklarını çıkar
roc_data = best_predictions.select(
    F.col("label"),
    vector_to_array(F.col("probability"))[1].alias("prob_positive")
).toPandas()

# ROC Curve hesapla
from sklearn.metrics import roc_curve, auc as sklearn_auc

fpr, tpr, thresholds = roc_curve(roc_data["label"], roc_data["prob_positive"])
roc_auc = sklearn_auc(fpr, tpr)

# ROC Curve çiz
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color="#3498db", linewidth=2.5, label=f"{best_model_name} (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.5, label="Random (AUC = 0.5)")

ax.fill_between(fpr, tpr, alpha=0.1, color="#3498db")
ax.set_xlabel("False Positive Rate (FPR)", fontsize=12)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=12)
ax.set_title(f"ROC Curve — {best_model_name}", fontsize=14)
ax.legend(loc="lower right", fontsize=12)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nROC AUC Score: {roc_auc:.4f}")

In [ ]:
# ── Tüm modellerin ROC Curve karşılaştırması ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

colors_roc = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#e74c3c"]

for i, (model_name, preds) in enumerate(all_predictions.items()):
    try:
        roc_df = preds.select(
            F.col("label"),
            vector_to_array(F.col("probability"))[1].alias("prob_positive")
        ).toPandas()
        
        fpr_i, tpr_i, _ = roc_curve(roc_df["label"], roc_df["prob_positive"])
        auc_i = sklearn_auc(fpr_i, tpr_i)
        ax.plot(fpr_i, tpr_i, color=colors_roc[i], linewidth=2, 
                label=f"{model_name} (AUC={auc_i:.4f})")
    except Exception as e:
        print(f"  {model_name}: ROC çizilemedi - {e}")

ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.5, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve Karşılaştırması — Tüm Modeller", fontsize=14)
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Özet ──────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("SONUÇ ÖZETİ")
print("="*70)
print(f"\n  En iyi model: {best_model_name}")
print(f"  AUC:          {best_auc:.4f}")
print(f"  Accuracy:     {results_df.iloc[0]['Accuracy']:.4f}")
print(f"  F1:           {results_df.iloc[0]['F1']:.4f}")
print(f"  Precision:    {results_df.iloc[0]['Precision']:.4f}")
print(f"  Recall:       {results_df.iloc[0]['Recall']:.4f}")
print(f"\n  MLflow Experiment: steam_reviews_classification")
print(f"  MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"  Toplam Run Sayısı: 5")
print("\n" + "="*70)
print("ML modelleme tamamlandı ✓")
print("="*70)

In [ ]:
# ── Spark oturumunu kapat ──────────────────────────────────────────────────────
spark.stop()
print("Spark oturumu kapatıldı. ✓")